In [24]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased', do_lower_case=True)

In [25]:
dict_size = 40000 #2000
#special_tokens = 2

#tokenizer = Tokenizer(BPE())
#tokenizer.normalizer = Lowercase()
#tokenizer.pre_tokenizer = Whitespace()
#trainer = BpeTrainer(vocab_size=dict_size)
#tokenizer.train_from_iterator(train_set_large["content"], trainer)
#tokenizer.save("bpe_tokenizer.json")

In [26]:
class GRUBased(nn.Module):
    def __init__(self):
        super(GRUBased, self).__init__()
        hidden_size=250
        n_layers=3
        
        self.gru = nn.GRU(hidden_size, hidden_size, n_layers)
        
        self.embeddings = nn.Embedding(num_embeddings=dict_size, embedding_dim=hidden_size)
        self.hidden = torch.zeros(n_layers, hidden_size, requires_grad=False)
        self.proj_out = nn.Linear(hidden_size, 1)

    def forward(self, seq):
        toks = torch.LongTensor(tokenizer.encode(seq)[:400])
        x = self.embeddings.forward(toks)
        #hidden = torch.zeros(self.n_layers, self.hidden_size, requires_grad=False)
        out, hidden = self.gru.forward(x, self.hidden)
        return self.proj_out.forward(hidden[-1]).tanh()

In [ ]:
from transformers import get_linear_schedule_with_warmup

model = GRUBased()
model.train()

learning_rate = 1e-5
adam_epsilon = 1e-8

opt = torch.optim.Adam(model.parameters(), lr=learning_rate, eps=adam_epsilon)
costs = []
epochs = 1

total_steps = len(train_set_large.index) * epochs
scheduler = get_linear_schedule_with_warmup(opt, num_warmup_steps=0, num_training_steps=total_steps)

for epoch in range(epochs):
    start_time = time.time()
    n = 0

    for seq, label in zip(train_set_large["content"], train_set_large["new_labels"]):    
        if label == "fake":
            y = 1
        else:
            y = -1
        
        out = model.forward(seq)
        loss = F.mse_loss(out, torch.Tensor([y]))
        costs.append(loss.item())
        loss.backward()
        opt.step()
        scheduler.step()
        opt.zero_grad()
        
        # Track time for every batch
        elapsed_time = time.time() - start_time
        batches_done = n + 1
        batches_total = len(train_set_large.index)
        
        # Estimate time remaining (ETA)
        remaining_batches = batches_total - batches_done
        eta_seconds = (elapsed_time / batches_done) * remaining_batches
        eta_str = str(time.strftime("%H:%M:%S", time.gmtime(eta_seconds)))
        
        # Print current status and ETA
        print(f"Epoch {epoch} : {n} / {len(train_set_large.index)} - "
              f"Loss: {float(np.mean(costs)):.2f} - ETA of epoch: {eta_str}", end='\r')
        
        n += 1

Token indices sequence length is longer than the specified maximum sequence length for this model (983 > 512). Running this sequence through the model will result in indexing errors


Epoch 0 : 20778 / 75026 - Loss: 0.59 - ETA of epoch: 01:50:15

In [ ]:
model.eval()

def evaluate_model(test_set, model_f):
    true_positive = 0
    false_negative = 0
    correct = 0
    n = 0
    
    for seq, label in zip(test_set["content"], test_set["new_labels"]):    
        y_hat = "fake" if model_f(seq) > 0.0 else "reliable"
        
        if y_hat == "fake" and label == "fake":
            true_positive += 1
        if y_hat == "fake" and label == "reliable":
            false_negative += 1
        if y_hat == label:
            correct += 1

        print(f"{n} / {len(test_set.index)}", end='\r')
        n += 1


    print(f"F1: {true_positive / (true_positive + false_negative) * 100}%")
    print(f"accuracy: {correct / len(test_set) * 100}%")

#print("training:")
#evaluate_model(train_set_large)
#
#print("\ntest time:")
#evaluate_model(test_set2)

In [ ]:
evaluate_model(test_set2, lambda seq: model.forward(seq).item())